In [1]:
# All import
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [2]:
# MODULE 1: Configuration
class Config:
    """
    Class quản lý tập trung toàn bộ đường dẫn và siêu tham số.
    Tuyệt đối không hardcode trong các module xử lý logic.
    """
    # 1. Đường dẫn dữ liệu (Kaggle environment)
    TRAIN_CSV_PATH = "/kaggle/input/datasets/mntrnlminh/fashion-mnist/fashion-mnist_train.csv"
    TEST_CSV_PATH = "/kaggle/input/datasets/mntrnlminh/fashion-mnist/fashion-mnist_test.csv"
    
    # 2. Cấu hình phân chia dữ liệu
    VAL_SPLIT_RATIO = 0.2
    RANDOM_STATE = 42
    
    # 3. Thông số dữ liệu
    IMAGE_SIZE = 28          # Kích thước ảnh gốc (28x28)
    NUM_CHANNELS = 3         # Đầu ra 3 kênh để phù hợp với ResNet/DenseNet
    NUM_CLASSES = 10         # 10 loại trang phục
    
    # 4. Siêu tham số huấn luyện (Chuẩn bị cho các Commit sau)
    BATCH_SIZE = 64
    EPOCHS = 20
    LEARNING_RATE = 1e-4
    GRAD_ACCUM_STEPS = 4     # Tích lũy đạo hàm 4 steps
    PATIENCE = 5             # Early Stopping patience

In [3]:
# MODULE 2: Dataset and Datapipeline
class FashionMNISTDataset(Dataset):
    """
    Class Dataset xử lý dữ liệu từ Pandas DataFrame.
    Bóc tách label, reshape pixels và nhân bản kênh ảnh.
    """
    def __init__(self, dataframe, transform=None):
        # Reset index để tránh lỗi khi lấy item sau khi split
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        
        # Bóc tách nhãn (cột đầu tiên thường là 'label') và pixel
        self.labels = self.dataframe['label'].values
        # Bỏ cột label, giữ lại 784 cột pixel
        self.pixels = self.dataframe.drop('label', axis=1).values
        
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        # 1. Lấy nhãn
        label = self.labels[idx]
        
        # 2. Xử lý ảnh: Reshape 784 pixels thành ma trận 28x28
        # Dữ liệu CSV có giá trị [0, 255], cần chuẩn hóa về [0, 1]
        image_1c = self.pixels[idx].reshape(Config.IMAGE_SIZE, Config.IMAGE_SIZE).astype(np.float32) / 255.0
        
        # 3. Trick cho Transfer Learning: Nhân bản ảnh 1 kênh thành 3 kênh (3, 28, 28)
        # Sử dụng np.stack để tạo 3 kênh giống hệt nhau
        image_3c = np.stack([image_1c, image_1c, image_1c], axis=0)
        
        # 4. Chuyển đổi thành PyTorch Tensor
        image_tensor = torch.tensor(image_3c, dtype=torch.float32)
        label_tensor = torch.tensor(label, dtype=torch.long)
        
        # 5. Áp dụng Data Augmentation (nếu có truyền vào)
        if self.transform:
            image_tensor = self.transform(image_tensor)
            
        return image_tensor, label_tensor

def prepare_dataloaders(config):
    """
    Hàm phụ trợ: Đọc CSV, chia Train/Val và khởi tạo DataLoader
    """
    print(f"[INFO] Đang đọc dữ liệu từ: {config.TRAIN_CSV_PATH}")
    # Có thể thêm khối try-except để mock data nếu test trên máy cá nhân không có Kaggle path
    try:
        df = pd.read_csv(config.TRAIN_CSV_PATH)
    except FileNotFoundError:
        print("[WARNING] Không tìm thấy file. Đang tạo dữ liệu giả lập để test code...")
        # Tạo mock dataframe nếu chạy test ngoài Kaggle
        mock_data = np.random.randint(0, 255, size=(1000, 785))
        mock_data[:, 0] = np.random.randint(0, 10, size=1000) # Cột label
        cols = ['label'] + [f'pixel{i}' for i in range(1, 785)]
        df = pd.DataFrame(mock_data, columns=cols)

    # Chia tập dữ liệu (stratify giúp cân bằng tỷ lệ class giữa Train và Val)
    train_df, val_df = train_test_split(
        df, 
        test_size=config.VAL_SPLIT_RATIO, 
        random_state=config.RANDOM_STATE,
        stratify=df['label']
    )
    
    print(f"[INFO] Chia tập dữ liệu thành công | Train: {len(train_df)} mẫu | Val: {len(val_df)} mẫu")
    
    # Khởi tạo Dataset
    train_dataset = FashionMNISTDataset(train_df)
    val_dataset = FashionMNISTDataset(val_df)
    
    # Khởi tạo DataLoader
    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=2)
    
    return train_loader, val_loader

In [4]:
# KHỐI THỰC THI KIỂM TRA
if __name__ == "__main__":
    # Khởi tạo config
    cfg = Config()
    
    # Chạy hàm tạo loader
    train_loader, val_loader = prepare_dataloaders(cfg)
    
    # Lấy thử 1 batch để kiểm tra
    images, labels = next(iter(train_loader))
    
    print("\n[KIỂM TRA KÍCH THƯỚC BATCH ĐẦU RA]")
    print(f"Kích thước tensor hình ảnh : {images.shape} -> (Batch Size, Channels, Height, Width)")
    print(f"Kích thước tensor nhãn     : {labels.shape} -> (Batch Size)")
    print(f"Giá trị pixel max/min      : {images.max().item():.2f} / {images.min().item():.2f}")

[INFO] Đang đọc dữ liệu từ: /kaggle/input/datasets/mntrnlminh/fashion-mnist/fashion-mnist_train.csv
[INFO] Chia tập dữ liệu thành công | Train: 48000 mẫu | Val: 12000 mẫu

[KIỂM TRA KÍCH THƯỚC BATCH ĐẦU RA]
Kích thước tensor hình ảnh : torch.Size([64, 3, 28, 28]) -> (Batch Size, Channels, Height, Width)
Kích thước tensor nhãn     : torch.Size([64]) -> (Batch Size)
Giá trị pixel max/min      : 1.00 / 0.00
